In [ ]:
import duckdb
import os
import shutil
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[0]))  # points to dlt/

In [ ]:
conn = duckdb.connect("../schemas/eyeon.duckdb")

In [ ]:
def get_columns(conn, schema, table_name, null_pct=0):
    # Note: null_percentage can be 100, and there still be values due to rounding.
    sql = f"select column_name, null_percentage from (summarize {schema}.{table_name}) where null_percentage > {null_pct}"
    columns = conn.sql(sql).fetchnumpy()
    return columns


columns = get_columns(conn, "silver", "raw_obs")

for name, pct in zip(columns["column_name"], columns["null_percentage"]):
    print(f"  Col: {name}  {pct}")

In [ ]:
def gen_mask_sql(conn, schema, table_name, null_pct=0):
    # Generate SQL for creating the mask summary for columns with nulls
    sql = """
    with row_mask as (
      select uuid,
        (
    """
    column_names = get_columns(conn, schema, table_name, null_pct)["column_name"]

    if column_names.size > 0:
        plus = ""
        for ndx, name in enumerate(column_names):
            if ndx < 128:
                sql += f"{plus} (({name} is not null)::uhugeint << {ndx})"
                plus = "+"

        sql += f"""
          ) mask0
        from {schema}.{table_name}
      )
      select
        mask0,
        count(*) as num_rows,
        first(uuid) as example_uuid
      from row_mask
      group by mask0
      order by num_rows
      """
    else:
        # All columns are filled! Just return any single row as our example.
        sql = f"select 0 as mask0, -1 num_rows, uuid example_uuid from {schema}.{table_name} limit 1"

    return sql


gen_mask_sql(conn, "silver", "metadata_pe_file")

In [ ]:
conn.sql(gen_mask_sql(conn, "silver", "metadata_pe_file")).show()

In [ ]:
def get_filenames(conn, schema, table):
    # Get the filenames...
    sql = f"""
    select o.source_file, o.source_path
    from silver.raw_obs o
    join ({gen_mask_sql(conn, schema, table)}) m on m.example_uuid=o.uuid
    """
    # print(sql)
    return conn.sql(sql).fetchnumpy()

In [ ]:
# Get all the filenames... for now, only tables with UUID are processed.
tables_with_uuid = conn.sql(
    "select schema_name, table_name from duckdb_columns() where column_name='uuid'"
).fetchnumpy()
dst_dir = "min_files_max_schema"

for schema, table in zip(
    tables_with_uuid["schema_name"], tables_with_uuid["table_name"]
):
    print(f"{schema} {table}")
    all_files = get_filenames(conn, schema, table)
    os.makedirs(dst_dir, exist_ok=True)
    for source_path, source_file in zip(
        all_files["source_path"], all_files["source_file"]
    ):
        # Copy file into an examples dir
        shutil.copy2(os.path.join(source_path, source_file), dst_dir)